# Financial Market Data Collection & Preprocessing

## Objective

The objective of this notebook is to build a clean and structured financial dataset for future quantitative finance research tasks including:

- Cointegration analysis
- Engle-Granger testing
- Johansen testing
- Spread analysis
- Rolling window stability analysis
- Statistical arbitrage research

The dataset consists of major US equities from the S&P 500 along with sector and market ETFs downloaded using Yahoo Finance.

## Import Libraries 

In [108]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [109]:
ETFS = [
    'XLK',   # Technology
    'XLF',   # Financials
    'XLV',   # Healthcare
    'XLY',   # Consumer Discretionary
    'XLP',   # Consumer Staples
    'XLE',   # Energy
    'XLI',   # Industrials
    'XLB',   # Materials
    'XLU',   # Utilities
    'XLRE',  # Real Estate
    'XLC',   # Communication Services
    'SPY'    # Broad Market
]

### Why Sector ETFs?

Individual stock universes suffer from:

- survivorship bias
- index membership changes
- mergers and delistings
- look-ahead selection bias

Sector ETFs provide stable representations of economic sectors through time.

This makes them suitable for long-horizon econometric analysis.

In [110]:
def download_adjusted_close(
    tickers,
    start_date,
    end_date
):
    """
    Download adjusted close prices.
    """

    try:

        data = yf.download(
            tickers,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=True
        )

        adj_close = data['Adj Close']

        return adj_close

    except Exception as e:

        raise RuntimeError(
            f"Download failed: {e}"
        )

In [111]:
START_DATE = "2018-01-01"

END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

adj_close = download_adjusted_close(
    ETFS,
    START_DATE,
    END_DATE
)

adj_close.head()

[*********************100%***********************]  12 of 12 completed


Ticker,SPY,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
2018-01-02,236.562180,25.995672,NaN,25.747259,23.884140,66.254852,29.872923,45.314568,24.837578,20.132584,72.723351,46.275246
2018-01-03,238.058395,26.177763,NaN,26.132858,24.012455,66.611717,30.122095,45.298534,24.845171,19.974421,73.419197,46.487705
2018-01-04,239.061752,26.406424,NaN,26.290604,24.234871,67.099136,30.274368,45.426765,24.420460,19.808548,73.523575,46.640121
2018-01-05,240.655014,26.618155,NaN,26.280085,24.303314,67.560448,30.592762,45.627140,24.473551,19.800833,74.149818,47.009617
2018-01-08,241.095032,26.656260,NaN,26.437836,24.269094,67.838974,30.708111,45.739323,24.640402,19.985998,73.880180,47.065033


Adjusted Close incorporates:

- stock splits
- reverse splits
- dividends
- distributions

Using raw close prices can create artificial jumps and distort return calculations.

Adjusted prices are therefore preferred for quantitative research.

In [112]:
def missing_summary(df):

    summary = pd.DataFrame({

        "Missing Count":
        df.isna().sum(),

        "Missing Percent":
        df.isna().mean() * 100
    })

    return summary.sort_values(
        "Missing Percent",
        ascending=False
    )

In [113]:
def check_duplicate_dates(df):

    return df.index.duplicated().sum()

In [114]:
def validate_alignment(df):

    return {
        "Start Date":
        df.index.min(),

        "End Date":
        df.index.max(),

        "Rows":
        len(df)
    }

In [115]:
missing_report = missing_summary(
    adj_close
)

missing_report

,Missing Count,Missing Percent
Ticker,,
XLC,116,5.492424
SPY,0,0.000000
XLB,0,0.000000
XLE,0,0.000000
XLF,0,0.000000
XLI,0,0.000000
XLK,0,0.000000
XLP,0,0.000000
XLRE,0,0.000000


In [116]:
duplicates = check_duplicate_dates(
    adj_close
)

print(
    f"Duplicate Dates: {duplicates}"
)

Duplicate Dates: 0


In [117]:
validate_alignment(
    adj_close
)

{'Start Date': Timestamp('2018-01-02 00:00:00'),
 'End Date': Timestamp('2026-05-28 00:00:00'),
 'Rows': 2112}

In [118]:
date_gaps = (
    adj_close.index
    .to_series()
    .diff()
    .value_counts()
)

date_gaps.head()

1 days    1649
3 days     381
4 days      57
2 days      24
Name: count, dtype: int64

In [119]:
trading_days = (
    adj_close.notna()
    .sum()
)

trading_days

Ticker
SPY     2112
XLB     2112
XLC     1996
XLE     2112
XLF     2112
XLI     2112
XLK     2112
XLP     2112
XLRE    2112
XLU     2112
XLV     2112
XLY     2112
dtype: int64

In [120]:
missing_report = pd.DataFrame({
    "Missing Count": adj_close.isna().sum(),
    "Missing Percent": adj_close.isna().mean()*100
})

missing_report.sort_values(
    "Missing Percent",
    ascending=False
)

,Missing Count,Missing Percent
Ticker,,
XLC,116,5.492424
SPY,0,0.000000
XLB,0,0.000000
XLE,0,0.000000
XLF,0,0.000000
XLI,0,0.000000
XLK,0,0.000000
XLP,0,0.000000
XLRE,0,0.000000


In [121]:
adj_close[adj_close['XLC'].isna()].head()

Ticker,SPY,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
2018-01-02,236.562180,25.995672,NaN,25.747259,23.884140,66.254852,29.872923,45.314568,24.837578,20.132584,72.723351,46.275246
2018-01-03,238.058395,26.177763,NaN,26.132858,24.012455,66.611717,30.122095,45.298534,24.845171,19.974421,73.419197,46.487705
2018-01-04,239.061752,26.406424,NaN,26.290604,24.234871,67.099136,30.274368,45.426765,24.420460,19.808548,73.523575,46.640121
2018-01-05,240.655014,26.618155,NaN,26.280085,24.303314,67.560448,30.592762,45.627140,24.473551,19.800833,74.149818,47.009617
2018-01-08,241.095032,26.656260,NaN,26.437836,24.269094,67.838974,30.708111,45.739323,24.640402,19.985998,73.880180,47.065033


In [122]:
adj_close[adj_close['XLC'].isna()].tail()

Ticker,SPY,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
2018-06-12,246.479187,25.929214,NaN,27.081482,23.996840,66.862686,33.377758,41.386154,24.587059,19.107185,74.323021,51.368080
2018-06-13,245.692596,25.648672,NaN,26.979103,23.910976,66.329605,33.206497,41.257191,24.037798,19.056652,74.340485,51.446842
2018-06-14,246.311157,25.686924,NaN,26.940266,23.687757,66.076187,33.442554,41.329735,24.266655,19.293793,74.724525,51.984192
2018-06-15,245.996948,25.507572,NaN,26.361637,23.686890,65.911232,33.314827,41.860912,24.227421,19.431067,74.869972,52.069656
2018-06-18,245.491028,25.443512,NaN,26.624651,23.643778,65.639221,33.351974,41.251770,24.204336,19.493824,74.151596,51.981400


In [123]:
print("First valid date:", adj_close['XLC'].first_valid_index())
print("Last valid date:", adj_close['XLC'].last_valid_index())

First valid date: 2018-06-19 00:00:00
Last valid date: 2026-05-28 00:00:00


In [124]:
adj_close = adj_close.dropna()

print(adj_close.shape)

print("Start Date:", adj_close.index.min())
print("End Date:", adj_close.index.max())

(1996, 12)
Start Date: 2018-06-19 00:00:00
End Date: 2026-05-28 00:00:00


In [125]:
missing_report = pd.DataFrame({
    "Missing Count": adj_close.isna().sum(),
    "Missing Percent": adj_close.isna().mean()*100
})

missing_report

,Missing Count,Missing Percent
Ticker,,
SPY,0,0.0
XLB,0,0.0
XLC,0,0.0
XLE,0,0.0
XLF,0,0.0
XLI,0,0.0
XLK,0,0.0
XLP,0,0.0
XLRE,0,0.0


In [126]:
log_prices = np.log(adj_close)

log_prices.head()

Ticker,SPY,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
2018-06-19,5.499420,3.217997,3.835667,3.279565,3.160544,4.162827,3.501951,3.724996,3.187802,2.979909,4.308707,3.949634
2018-06-20,5.501124,3.214743,3.848000,3.283971,3.157981,4.163510,3.504048,3.725975,3.198538,2.980705,4.310826,3.954365
2018-06-21,5.494836,3.204051,3.841853,3.265281,3.155045,4.150876,3.496335,3.727929,3.204488,2.984084,4.305048,3.947216
2018-06-22,5.496657,3.218510,3.846220,3.285036,3.150254,4.154325,3.493092,3.736097,3.213192,2.991005,4.309532,3.945511
2018-06-25,5.482951,3.202837,3.825406,3.264738,3.139483,4.141573,3.472111,3.741120,3.210713,3.007421,4.300305,3.923532


In [127]:
returns = adj_close.pct_change().dropna()

returns.head()

Ticker,SPY,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
2018-06-20,0.001706,-0.003249,0.012410,0.004416,-0.002559,0.000683,0.002100,0.000979,0.010794,0.000797,0.002121,0.004742
2018-06-21,-0.006269,-0.010634,-0.006129,-0.018517,-0.002932,-0.012555,-0.007684,0.001956,0.005967,0.003385,-0.005762,-0.007123
2018-06-22,0.001823,0.014563,0.004376,0.019952,-0.004780,0.003455,-0.003238,0.008202,0.008742,0.006945,0.004494,-0.001704
2018-06-25,-0.013612,-0.015551,-0.020598,-0.020093,-0.010713,-0.012670,-0.020762,0.005036,-0.002476,0.016552,-0.009185,-0.021739
2018-06-26,0.002214,0.003819,0.001658,0.012629,-0.003361,0.003766,0.004038,-0.004240,0.005275,0.001163,-0.003090,0.007162


In [128]:
log_returns = log_prices.diff().dropna()

log_returns.head()

Ticker,SPY,XLB,XLC,XLE,XLF,XLI,XLK,XLP,XLRE,XLU,XLV,XLY
2018-06-20,0.001705,-0.003254,0.012334,0.004406,-0.002562,0.000683,0.002098,0.000979,0.010736,0.000796,0.002119,0.004730
2018-06-21,-0.006288,-0.010691,-0.006148,-0.018690,-0.002937,-0.012634,-0.007713,0.001955,0.005950,0.003379,-0.005778,-0.007149
2018-06-22,0.001821,0.014458,0.004367,0.019755,-0.004791,0.003449,-0.003243,0.008168,0.008704,0.006921,0.004484,-0.001705
2018-06-25,-0.013706,-0.015673,-0.020813,-0.020298,-0.010771,-0.012751,-0.020981,0.005023,-0.002479,0.016416,-0.009227,-0.021979
2018-06-26,0.002212,0.003812,0.001657,0.012550,-0.003366,0.003759,0.004030,-0.004249,0.005261,0.001163,-0.003095,0.007137


In [129]:
print("Adjusted Close Shape:", adj_close.shape)
print("Log Prices Shape:", log_prices.shape)
print("Returns Shape:", returns.shape)
print("Log Returns Shape:", log_returns.shape)

Adjusted Close Shape: (1996, 12)
Log Prices Shape: (1996, 12)
Returns Shape: (1995, 12)
Log Returns Shape: (1995, 12)


In [130]:
adj_close.describe().T

,count,mean,std,min,25%,50%,75%,max
Ticker,,,,,,,,
SPY,1996.0,417.102702,134.913548,204.944839,297.531815,396.405807,514.945282,754.599976
XLB,1996.0,35.737548,8.158101,17.023155,26.650621,37.726139,42.168242,53.382599
XLC,1996.0,69.161754,22.625750,36.349846,49.372574,64.110867,81.396080,119.698029
XLE,1996.0,31.856188,11.449025,9.245276,22.261572,33.374472,41.479834,62.560001
XLF,1996.0,34.243077,10.092721,15.840228,24.832751,32.812107,40.264023,56.111568
XLI,1996.0,99.200374,30.689325,44.495220,71.842913,94.703907,119.870893,178.398712
XLK,1996.0,77.371327,34.413762,26.972515,48.706739,71.465843,102.696838,186.850006
XLP,1996.0,63.860482,11.882121,40.222927,53.513637,65.784492,73.316374,89.506195
XLRE,1996.0,34.758292,5.423533,20.815006,30.453609,34.777737,39.970621,44.843876


In [131]:
returns.describe().T

,count,mean,std,min,25%,50%,75%,max
Ticker,,,,,,,,
SPY,1995.0,0.000639,0.012186,-0.109424,-0.004265,0.000899,0.006504,0.105019
XLB,1995.0,0.000458,0.013878,-0.110084,-0.006883,0.000710,0.007880,0.117601
XLC,1995.0,0.000561,0.013989,-0.112795,-0.005665,0.001107,0.007629,0.089901
XLE,1995.0,0.000587,0.020137,-0.201412,-0.009112,0.001061,0.010299,0.160373
XLF,1995.0,0.000499,0.014780,-0.137093,-0.005897,0.000843,0.007502,0.131566
XLI,1995.0,0.000590,0.013491,-0.113441,-0.005562,0.000924,0.007335,0.126512
XLK,1995.0,0.001006,0.016694,-0.138140,-0.007186,0.001756,0.009905,0.134257
XLP,1995.0,0.000405,0.009818,-0.093956,-0.004220,0.000604,0.005375,0.085106
XLRE,1995.0,0.000400,0.013820,-0.160012,-0.006143,0.000961,0.007316,0.087800


In [132]:
metadata = pd.DataFrame({

    "ETF": [
        "XLK","XLF","XLV","XLY",
        "XLP","XLE","XLI","XLB",
        "XLU","XLRE","XLC","SPY"
    ],

    "Sector": [
        "Technology",
        "Financials",
        "Healthcare",
        "Consumer Discretionary",
        "Consumer Staples",
        "Energy",
        "Industrials",
        "Materials",
        "Utilities",
        "Real Estate",
        "Communication Services",
        "Broad Market"
    ]
})

metadata

,ETF,Sector
0,XLK,Technology
1,XLF,Financials
2,XLV,Healthcare
3,XLY,Consumer Discretionary
4,XLP,Consumer Staples
5,XLE,Energy
6,XLI,Industrials
7,XLB,Materials
8,XLU,Utilities
9,XLRE,Real Estate


In [133]:
from pathlib import Path

output_dir = Path(
    "../data/processed"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [136]:
adj_close.to_csv(
    output_dir / "adjusted_close.csv"
)

log_prices.to_csv(
    output_dir / "log_prices.csv"
)

returns.to_csv(
    output_dir / "returns.csv"
)

log_returns.to_csv(
    output_dir / "log_returns.csv"
)

In [135]:
print("="*50)

print("Adjusted Close:", adj_close.shape)

print("Log Prices:", log_prices.shape)

print("Returns:", returns.shape)

print("Log Returns:", log_returns.shape)

print()

print("Missing Values:")
print(adj_close.isna().sum().sum())

print()

print("Start Date:", adj_close.index.min())
print("End Date:", adj_close.index.max())

print("="*50)

Adjusted Close: (1996, 12)
Log Prices: (1996, 12)
Returns: (1995, 12)
Log Returns: (1995, 12)

Missing Values:
0

Start Date: 2018-06-19 00:00:00
End Date: 2026-05-28 00:00:00
